# Entrenamiento limpio de modelos de resultados de ajedrez

Lectura de partidas, entrenamiento de un árbol y un random forest, y serialización de ambos modelos.

In [1]:
from io import StringIO, TextIOWrapper
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import joblib
import numpy as np
import pandas as pd
import zstandard as zstd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier

In [ ]:
PGN_ZST_PATH = Path("DB/lichess_db_standard_rated_2013-01.pgn.zst")
STOCKFISH_PATH = Path("stockfish/stockfish")
STOCKFISH_DEPTH = 8
MOVIMIENTOS_DESDE_FINAL = 10
NUMERO_OBSERVACIONES_PARA_ENTRENO = 5000

RESULTADO_A_CLASE = {"1-0": 0, "0-1": 1, "1/2-1/2": 2}
VALOR_PIEZA = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
}
CARACTERISTICAS = [
    "reina_blancas", "reina_negras", "blancas_enrocadas",
    "negras_enrocadas", "dos_alfiles_blancas", "dos_alfiles_negras",
    "peon_pasado_blancas", "peon_pasado_negras", "mas_piezas_blancas",
    "mas_piezas_negras", "mas_peones_blancas", "mas_peones_negras",
    "ventaja_material", "movilidad_blancas", "movilidad_negras",
    "torre_adelantada_blancas", "torre_adelantada_negras",
    "evaluacion_stockfish",
]

In [3]:
def partidas_en_stream(ruta):
    with ruta.open("rb") as archivo:
        with zstd.ZstdDecompressor().stream_reader(archivo) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")
            while partida := chess.pgn.read_game(flujo_texto):
                yield partida

def tiene_peon_pasado(tablero, color):
    peones = tablero.pieces(chess.PAWN, color)
    rivales = tablero.pieces(chess.PAWN, not color)
    for casilla in peones:
        archivo = chess.square_file(casilla)
        rango = chess.square_rank(casilla)
        bloqueado = any(
            abs(chess.square_file(rival) - archivo) <= 1
            and (chess.square_rank(rival) > rango if color else chess.square_rank(rival) < rango)
            for rival in rivales
        )
        if not bloqueado:
            return True
    return False

def tiene_torre_adelantada(tablero, color):
    filas = {6, 7} if color else {0, 1}
    return any(chess.square_rank(casilla) in filas for casilla in tablero.pieces(chess.ROOK, color))

def movimientos_legales(tablero, color):
    copia = tablero.copy()
    copia.turn = color
    return copia.legal_moves.count()

def evaluacion_stockfish(tablero, motor):
    info = motor.analyse(tablero, chess.engine.Limit(depth=STOCKFISH_DEPTH))
    puntuacion = info["score"].pov(chess.WHITE).score(mate_score=10000)
    return float(np.clip(puntuacion / 100, -100, 100))

def estado_tablero(tablero, blancas_enrocadas, negras_enrocadas, motor):
    piezas_blancas = sum(len(tablero.pieces(tipo, chess.WHITE)) for tipo in range(chess.PAWN, chess.KING))
    piezas_negras = sum(len(tablero.pieces(tipo, chess.BLACK)) for tipo in range(chess.PAWN, chess.KING))
    peones_blancos = len(tablero.pieces(chess.PAWN, chess.WHITE))
    peones_negros = len(tablero.pieces(chess.PAWN, chess.BLACK))
    ventaja_material = sum(VALOR_PIEZA[pieza.piece_type] if pieza.color else -VALOR_PIEZA[pieza.piece_type] for pieza in tablero.piece_map().values())
    return {
        "reina_blancas": int(bool(tablero.pieces(chess.QUEEN, chess.WHITE))),
        "reina_negras": int(bool(tablero.pieces(chess.QUEEN, chess.BLACK))),
        "blancas_enrocadas": int(blancas_enrocadas), "negras_enrocadas": int(negras_enrocadas),
        "dos_alfiles_blancas": int(len(tablero.pieces(chess.BISHOP, chess.WHITE)) >= 2),
        "dos_alfiles_negras": int(len(tablero.pieces(chess.BISHOP, chess.BLACK)) >= 2),
        "peon_pasado_blancas": int(tiene_peon_pasado(tablero, chess.WHITE)),
        "peon_pasado_negras": int(tiene_peon_pasado(tablero, chess.BLACK)),
        "mas_piezas_blancas": int(piezas_blancas > piezas_negras), "mas_piezas_negras": int(piezas_negras > piezas_blancas),
        "mas_peones_blancas": int(peones_blancos > peones_negros), "mas_peones_negras": int(peones_negros > peones_blancos),
        "ventaja_material": ventaja_material,
        "movilidad_blancas": movimientos_legales(tablero, chess.WHITE), "movilidad_negras": movimientos_legales(tablero, chess.BLACK),
        "torre_adelantada_blancas": int(tiene_torre_adelantada(tablero, chess.WHITE)),
        "torre_adelantada_negras": int(tiene_torre_adelantada(tablero, chess.BLACK)),
        "evaluacion_stockfish": evaluacion_stockfish(tablero, motor),
    }

def partida_a_dataframe(partida, movimientos_desde_final, motor):
    resultado = partida.headers.get("Result", "*")
    if resultado not in RESULTADO_A_CLASE:
        raise ValueError(f"Resultado PGN no válido: {resultado}")
    movimientos = list(partida.mainline_moves())
    indice_objetivo = max(0, len(movimientos) - movimientos_desde_final * 2)
    tablero = partida.board()
    blancas_enrocadas = negras_enrocadas = False
    for indice, movimiento in enumerate(movimientos):
        if tablero.is_castling(movimiento):
            if tablero.turn == chess.WHITE:
                blancas_enrocadas = True
            else:
                negras_enrocadas = True
        tablero.push(movimiento)
        if indice + 1 == indice_objetivo:
            break
    caracteristicas = estado_tablero(tablero, blancas_enrocadas, negras_enrocadas, motor)
    caracteristicas["y"] = RESULTADO_A_CLASE[resultado]
    return pd.DataFrame([caracteristicas], columns=CARACTERISTICAS + ["y"])

def extrae_caracteristicas(ruta, numero_observaciones, movimientos_desde_final):
    datos = []
    claves = set()
    motor = chess.engine.SimpleEngine.popen_uci(str(STOCKFISH_PATH))
    try:
        for partida in partidas_en_stream(ruta):
            fila = partida_a_dataframe(partida, movimientos_desde_final, motor)
            clave = tuple(fila[CARACTERISTICAS].iloc[0])
            if clave not in claves:
                claves.add(clave)
                datos.append(fila)
            if len(datos) >= numero_observaciones:
                break
    finally:
        motor.quit()
    return pd.concat(datos, ignore_index=True) if datos else pd.DataFrame(columns=CARACTERISTICAS + ["y"])

In [4]:
datos_observaciones = extrae_caracteristicas(
    PGN_ZST_PATH, NUMERO_OBSERVACIONES_PARA_ENTRENO, MOVIMIENTOS_DESDE_FINAL
)
X = datos_observaciones[CARACTERISTICAS]
y = datos_observaciones["y"]
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Observaciones: {len(datos_observaciones)}")

Observaciones: 5000


In [5]:
param_grid_arbol = {
    "max_depth": [3, 5, 7, 10, None],
    "max_features": [None, "sqrt", "log2"],
    "min_samples_split": [2, 5, 10, 20],
}
grid_search_arbol = GridSearchCV(
    DecisionTreeClassifier(random_state=42), param_grid_arbol,
    scoring="accuracy", cv=5, n_jobs=-1
)
grid_search_arbol.fit(X_train, y_train)
arbol_optimizado = grid_search_arbol.best_estimator_
precision_arbol = accuracy_score(y_validation, arbol_optimizado.predict(X_validation))
print("Mejores parámetros del árbol:", grid_search_arbol.best_params_)
print(f"Precisión CV: {grid_search_arbol.best_score_:.2%}")
print(f"Precisión de validación: {precision_arbol:.2%}")

Mejores parámetros del árbol: {'max_depth': 7, 'max_features': None, 'min_samples_split': 20}
Precisión CV: 68.65%
Precisión de validación: 69.30%


In [6]:
param_grid_bosque = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", "log2", None],
    "min_samples_split": [2, 5, 10, 20],
}
grid_search_bosque = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1), param_grid_bosque,
    scoring="accuracy", cv=5, n_jobs=-1
)
grid_search_bosque.fit(X_train, y_train)
bosque_aleatorio = grid_search_bosque.best_estimator_
precision_bosque = accuracy_score(y_validation, bosque_aleatorio.predict(X_validation))
print("Mejores parámetros del bosque:", grid_search_bosque.best_params_)
print(f"Precisión CV: {grid_search_bosque.best_score_:.2%}")
print(f"Precisión de validación: {precision_bosque:.2%}")

Mejores parámetros del bosque: {'max_depth': 10, 'max_features': None, 'min_samples_split': 20, 'n_estimators': 200}
Precisión CV: 70.45%
Precisión de validación: 72.20%


In [7]:
paquete_modelos = {
    "arbol": arbol_optimizado,
    "bosque": bosque_aleatorio,
    "caracteristicas": CARACTERISTICAS,
    "resultado_a_clase": RESULTADO_A_CLASE,
    "stockfish_depth": STOCKFISH_DEPTH,
}
ruta_modelos = Path("modelo_chess_arbol_y_random_forest.joblib")
joblib.dump(paquete_modelos, ruta_modelos)
print(f"Modelos guardados en: {ruta_modelos}")

Modelos guardados en: modelo_chess_arbol_y_random_forest.joblib
